<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/%EC%A7%88%EC%9D%98_%EB%B3%80%ED%98%95_%EB%A9%80%ED%8B%B0_%EC%BF%BC%EB%A6%AC%2C_Hyde.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#환경설정

In [1]:
!pip install -U langchain langchain_openai langchain_community chromadb pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0

In [2]:
import os

os.environ['OPENAI_API_KEY'] = None

In [3]:
import logging

logging.basicConfig()
logging.getLogger('langchain.retrievers.multi_query').setLevel(logging.INFO)

#데이터 불러오기

In [5]:
!wget "https://github.com/chatgpt-kr/openai-api-tutorial/raw/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf"

--2026-03-01 04:04:09--  https://github.com/chatgpt-kr/openai-api-tutorial/raw/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/chatgpt-kr/openai-api-tutorial/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf [following]
--2026-03-01 04:04:09--  https://raw.githubusercontent.com/chatgpt-kr/openai-api-tutorial/main/ch06/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2242511 (2.1M) [application/octet-stream]
Sav

데이터불러오기

In [14]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('2023_북한인권보고서.pdf')
pages = loader.load_and_split()

print('pages : ', len(pages))

pages :  445


Chunking후 VectorDB 적제

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

recursive_splitter = RecursiveCharacterTextSplitter()
split_docs = recursive_splitter.split_documents(pages)

embeddings = OpenAIEmbeddings()

vector_db = Chroma.from_documents(documents = split_docs, embedding = embeddings)

#다중 질의 생성 리트리버 설정

In [99]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

import re, ast, logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('langchain.retrievers.multi_query')

class MultiQueryRetriever:

  def __init__(self, retriever, llm: ChatOpenAI, num_queries : int = 2):
    self.retriever = retriever
    self.llm = llm
    self.num_queries = num_queries
    self.prompt_template = ChatPromptTemplate.from_template(
    """
    당신은 전문적인 정보 검색 보조 멀티 쿼리 출력 시스템입니다.
    아래의 사용자의 질문을 서로 다른 관점에서 표현한 {num_queries}개의 검색 쿼리를 만드세요.

    #출력 형식
    반드시 아래 예시처럼 파이썬 문자열을 담은 리스트로만 출력하세요.
    절대로 백틱(''')이나 JSON 코드블록을 사용하지 마세요.

    #예시
    ['북한의 조직 생활 형태', '북한의 강제 조직 활동', '북한의 주민 조직 생활']

    #질문
    {question}
    """
    )


  #LLM 출력 파싱
  def _parse_list(self, text: str):
      text = re.sub(r"```.*?```", "", text, flags=re.DOTALL).strip()
      text = text.replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'")
      try:
          parsed = ast.literal_eval(text)
          if isinstance(parsed, list) and all(isinstance(x, str) for x in parsed):
              return [x.strip() for x in parsed if x.strip()]
      except Exception:
          pass
      lines = [re.sub(r"^[\d\.\-\* ]+", "", l).strip() for l in text.split("\n") if l.strip()]
      return lines[:self.num_queries]

  def generate_queries(self, question : str) -> list[str]:
    prompt = self.prompt_template.format_messages(question = question, num_queries = self.num_queries)
    response = llm.invoke(prompt)
    queries = self._parse_list(response.content)

    logger.info(f'Generated queries : {queries}')

    return queries

  def invoke(self, question : str) -> list[Document]:
    queries = self.generate_queries(question)

    seen = set()
    all_docs = []

    for q in queries:
      docs = self.retriever.invoke(q)
      for d in docs:
        if d.page_content not in seen:
          seen.add(d.page_content)
          all_docs.append(d)
    return queries, all_docs


In [100]:
llm = ChatOpenAI(model = 'gpt-4.1', temperature = 0.2)

mq_retriever = MultiQueryRetriever(
    retriever = vector_db.as_retriever(search_kwargs = {'k' : 2}),
    llm = llm,
    num_queries = 3
)

In [94]:
question = '북한의 현황'
result = mq_retriever.invoke(question)

INFO:langchain.retrievers.multi_query:Generated queries : ['북한의 정치, 경제, 사회 현황', '최근 북한의 변화와 동향', '2024년 기준 북한의 주요 이슈와 상황']


#RAG 챗봇 구현

In [101]:
result = mq_retriever.invoke('북한 현황')

INFO:langchain.retrievers.multi_query:Generated queries : ['북한의 정치, 경제, 사회 현황', '최근 북한의 변화와 동향', '2024년 기준 북한의 주요 이슈']


In [103]:
result[0]

['북한의 정치, 경제, 사회 현황', '최근 북한의 변화와 동향', '2024년 기준 북한의 주요 이슈']

In [106]:
def multi_query_rag(question):

  question_list, _ = mq_retriever.invoke(question)
  llm = ChatOpenAI(model = 'gpt-4.1', temperature = 0.2)

  prompt_template = ChatPromptTemplate.from_template("""
당신은 북한 전문가입니다.
다음 검색 결과와 질문을 참고하여 적절한 질문을 하세요.
검색 결과에 질문에 대한 답이 없을 경우 모른다고 솔직하게 대답하세요

#검색 결과
{context}

#질문
{question}
"""
)
  print('question_list : ', question_list)
  print('multi query length : ', len(question_list))
  response_list = []

  for q in question_list:
    docs = vector_db.similarity_search(q)

    context = "".join([doc.page_content for doc in docs])

    query = prompt_template.format_messages(context = context, question = q)
    response = llm.invoke(q)

    response_list.append(response)

  return response_list

In [107]:
answer = multi_query_rag("북한 사람들의 현황")

INFO:langchain.retrievers.multi_query:Generated queries : ['북한 주민의 생활 실태', '북한 인구의 사회적, 경제적 현황', '북한 사람들의 일상과 삶의 조건']


question_list :  ['북한 주민의 생활 실태', '북한 인구의 사회적, 경제적 현황', '북한 사람들의 일상과 삶의 조건']
multi query length :  3


In [109]:
for a in answer:
  print(a.content)

북한 주민의 생활 실태는 외부에 공개된 정보가 제한적이지만, 탈북자 증언, 위성사진, 국제기구 및 언론 보도 등을 통해 어느 정도 파악할 수 있습니다. 아래에 주요 내용을 정리합니다.

### 1. 경제 상황
- **배급제**: 북한은 공식적으로 국가가 식량과 생필품을 배급하는 ‘공급제’를 유지하고 있으나, 1990년대 중반 ‘고난의 행군’ 이후 배급이 제대로 이루어지지 않고 있습니다. 많은 주민들이 장마당(비공식 시장)에서 생계를 유지합니다.
- **장마당 경제**: 주민들은 장마당에서 식료품, 의류, 생필품 등을 사고팝니다. 최근에는 휴대전화, 중국산 전자제품 등도 거래됩니다.
- **빈부격차**: 평양 등 일부 대도시와 지방, 당 간부와 일반 주민 간의 경제적 격차가 심화되고 있습니다.

### 2. 식량 및 영양 상태
- **식량난**: 만성적인 식량 부족 상태입니다. 쌀, 옥수수 등 곡물 배급이 부족해 영양실조, 발육부진 문제가 심각합니다.
- **국제 지원**: 유엔 등 국제기구의 식량 지원에 의존하는 경우가 많으나, 최근 코로나19로 국경이 봉쇄되면서 지원도 크게 줄었습니다.

### 3. 주거 및 생활 환경
- **주택**: 평양 등 대도시의 일부 아파트는 현대화되어 있으나, 지방이나 농촌은 낙후된 주택이 많습니다.
- **전기 및 수도**: 전력 공급이 불안정해 정전이 잦고, 수도 공급도 원활하지 않은 경우가 많습니다.

### 4. 교육 및 의료
- **교육**: 무상교육이 원칙이나, 실질적으로는 교사에게 뇌물을 주거나, 학생들이 농사 등 노동에 동원되는 경우가 많습니다.
- **의료**: 무상의료를 표방하지만, 약품과 의료기기가 부족해 실질적인 치료가 어렵습니다. 의료서비스를 받으려면 뇌물이나 현물을 요구받는 경우도 있습니다.

### 5. 사회 통제와 인권
- **감시와 통제**: 주민들은 국가의 엄격한 감시와 통제 하에 생활합니다. 이동의 자유가 제한되고, 외부 정보(남한 드라마, 외국 뉴스 등) 유입이 엄격히 금지되어 있습니다.
- *

In [29]:
llm = ChatOpenAI(model = 'gpt-4.1', temperature = 0.2)

prompt_template = """
당신은 북한 전문가입니다.
다음 검색 결과와 질문을 참고하여 적절한 질문을 하세요.
검색 결과에 질문에 대한 답이 없을 경우 모른다고 솔직하게 대답하세요

#검색 결과
{context}

#질문
{question}
"""

#가상 문서 임베딩

In [110]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

#PDF 불러오기
loader = PyPDFLoader('2023_북한인권보고서.pdf')
pages = loader.load_and_split()

#불러온 PDF 자르기
recursive_spltiter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = recursive_splitter.split_documents(pages)

#자른 PDF VectorDB 적제하기
vector_db = Chroma.from_documents(documents=splitted_docs, embedding = OpenAIEmbeddings())
retriever = vector_db.as_retriever()

가상 문서 생성 함수

In [112]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

def generate_virtual_doc(inputs): #inputs 딕셔너리
  question = inputs['question']
  prompt_template = ChatPromptTemplate.from_template(
      """
      당신은 고도로 숙련된 AI입니다.
      아래의 질문에 직접적인 답변을 제시하는 짧은 문서를 작성하세요.
      문서의 길이는 약 200자 내외로 하세요.

      질문: {question}
      """
  )
  llm = ChatOpenAI(model = 'gpt-4.1', temperature = 0.1)

  prompt = prompt_template.format_messages(question = question)

  result = llm.invoke(prompt)

  return {'question' : question, 'virtual_doc' : result.content}

In [114]:
generate_virtual_doc({'question' : '이디야 커피'})

{'question': '이디야 커피',
 'virtual_doc': '이디야 커피는 2001년에 설립된 대한민국의 대표적인 커피 프랜차이즈입니다. 합리적인 가격과 다양한 메뉴, 전국적인 매장망을 바탕으로 빠르게 성장했습니다. 특히 아메리카노, 카페라떼 등 기본 메뉴 외에도 시즌별 신메뉴와 디저트, 베이커리 등 다양한 상품을 제공합니다. 접근성과 가성비를 중시하는 소비자들에게 인기가 높으며, 최근에는 배달 서비스와 자체 앱을 통한 멤버십 혜택도 강화하고 있습니다.'}

생성된 가상문서를 사용해 vectorDB 검색

In [120]:
def retrieve_documents(inputs): #inputs 딕셔너리
  virtual_doc = inputs['virtual_doc']
  docs = retriever.invoke(virtual_doc)

  return {**inputs, "retrieved_docs" : docs}

In [121]:
inputs = {'question' : '이디야 커피'}
print(inputs)

inputs = generate_virtual_doc(inputs)
print(inputs)

inputs = retrieve_documents(inputs)
print(inputs)

{'question': '이디야 커피'}
{'question': '이디야 커피', 'virtual_doc': '이디야 커피는 2001년에 설립된 대한민국의 대표적인 커피 프랜차이즈입니다. 합리적인 가격과 다양한 메뉴, 전국적인 매장망을 바탕으로 빠르게 성장했습니다. 특히 저렴한 가격과 접근성, 꾸준한 신메뉴 출시로 대학생과 직장인 등 다양한 고객층에게 인기가 많습니다.'}
{'question': '이디야 커피', 'virtual_doc': '이디야 커피는 2001년에 설립된 대한민국의 대표적인 커피 프랜차이즈입니다. 합리적인 가격과 다양한 메뉴, 전국적인 매장망을 바탕으로 빠르게 성장했습니다. 특히 저렴한 가격과 접근성, 꾸준한 신메뉴 출시로 대학생과 직장인 등 다양한 고객층에게 인기가 많습니다.', 'retrieved_docs': [Document(metadata={'page': 398, 'creator': 'Adobe InDesign CS6 (Windows)', 'trapped': '/False', 'total_pages': 448, 'moddate': '2023-07-31T13:57:54+09:00', 'producer': 'Adobe PDF Library 10.0.1', 'page_label': '399', 'source': '2023_북한인권보고서.pdf', 'creationdate': '2023-07-31T13:50:27+09:00'}, page_content='2. 아동\n397\nIV. 경제적·사회적·문화적 권리 I. 발간개요V. 취약계층VI. 특별사안 II. 요약III. 시민적·정치적 권리\n증언도 있었다. 2017년부터 2019년까지 중등학원 조리사로 일했다\n는 증언자는 북한 당국이 초등학원에 식재료를 지원하여 급식의 질\n이 향상되었다고 진술했다.\n“초등학원은 국가에서 고아들을 돌봐주는 곳으로 8살부터 12살까\n지 들어와서 소학교 과정을 배우면서 생활하는 곳입니다. 2017년에 \n고아시설이 김정은 방침에 의해 도마다 새 건물로 꾸려졌

context(문서 검색)

In [126]:
def build_context(inputs):
  docs = inputs['retrieved_docs']
  context = "\n\n".join(doc.page_content for doc in docs)

  return {**inputs, 'context' : context}

검색 함수

In [130]:
def generate_final_answer(inputs):
  question = inputs['question']
  context = inputs['context']

  prompt_template = ChatPromptTemplate.from_template(
        f"""
다음 정보를 참고하여 질문에 대한 답변을 작성하세요.

컨텍스트:
{context}

질문:
{question}

답변:
""".strip()
    )

  llm = ChatOpenAI(model = 'gpt-4.1', temperature = 0.1)
  prompt = prompt_template.format_messages()
  response = llm.invoke(prompt)

  return {**inputs, 'final_answer' : response.content}

In [131]:
# 전체 실행
def pipeline(question):
    inputs = {"question": question}
    step1 = generate_virtual_doc(inputs)
    step2 = retrieve_documents(step1)
    step3 = build_context(step2)
    step4 = generate_final_answer(step3)
    print(f"\n최종 답변: {step4['final_answer']}")
    return step4

In [132]:
pipeline('북한에 관하여')


최종 답변: 북한에 관하여

북한에서는 시민적·정치적 권리가 심각하게 제한되고 있습니다. 공정한 재판을 받을 권리가 제대로 보장되지 않으며, 행정기관이 법원의 역할을 대신해 노동교양형 등 징역형에 해당하는 처벌을 내릴 수 있습니다. 반국가·반민족 범죄로 간주되는 경우에는 법원의 재판 없이 정치범수용소에 수용되기도 합니다. 북한의 사법부는 조선노동당의 통제 아래에 있어 법원의 독립성이 인정되지 않으며, 공개재판 제도는 주민 교양을 위한 선전도구로 활용되고 있습니다. 수백 명 앞에서 공개재판이 이루어지거나, 피의자가 많은 사람들 앞에서 자신의 범죄를 인정하도록 강요받는 공개폭로모임도 존재합니다.

피고인의 권리인 변호권, 진술거부권, 상소권 등도 충분히 보장되지 않습니다. 변호인이 선임되더라도 변호인접견권이 제한되거나, 변호인이 실질적인 변론을 하지 않는 경우가 많다는 증언이 있습니다.

북한 주민들은 어릴 때부터 당국의 영향력 아래 사회조직에 소속되어 광범위한 감시와 통제를 받습니다. 인민반 제도는 주민을 감시하는 가장 하부조직으로, 인민반장과 통보원, 정보원 등이 주민들의 생활과 사상동향, 외부 방문자 등을 감시합니다. 특히 탈북 경험이 있거나 그 가족들은 더욱 엄격한 감시 대상이 됩니다.

종교의 자유 역시 명목상으로만 존재하며, 실제로는 보장되지 않습니다. 북한 당국의 지속적인 종교 탄압 정책으로 인해 대부분의 주민이 종교를 접해보지 못했고, 반종교 교육이 이루어집니다. 성경 소지나 선교 활동을 이유로 공개처형되거나 정치범수용소로 보내졌다는 증언도 있습니다. 미신행위도 비사회주의적 행위로 간주되어 단속과 처벌의 대상이 되며, 2018년 이후에는 처벌이 더욱 강화되었습니다.

언론·출판의 자유도 제한되어 있습니다. 당국의 지시에 따라 언론과 출판 내용이 정해지며, 반동적인 사상과 문화, 생활풍조를 퍼뜨릴 수 있는 출판물은 법에 따라 회수됩니다. 최고지도자나 정치체제를 비난하는 ‘말반동’ 행위는 체포, 행방불명, 정치범수용소 수용 등으로 이어질 수 있습니다.

{'question': '북한에 관하여',
 'virtual_doc': '북한은 한반도 북부에 위치한 사회주의 국가로, 공식 명칭은 조선민주주의인민공화국입니다. 1948년 김일성에 의해 수립되었으며, 현재 김정은이 최고지도자입니다. 평양이 수도이고, 중앙집권적 1당 체제와 강력한 군사력을 유지하고 있습니다. 경제는 국가 주도로 운영되며, 국제사회와의 긴장과 제재로 인해 경제적 어려움을 겪고 있습니다.',
 'retrieved_docs': [Document(metadata={'creationdate': '2023-07-31T13:50:27+09:00', 'moddate': '2023-07-31T13:57:54+09:00', 'page_label': '29', 'source': '2023_북한인권보고서.pdf', 'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'page': 28, 'total_pages': 448, 'trapped': '/False'}, page_content='1. 시민적·정치적 권리\n27\nIV. 경제적·사회적·문화적 권리 I. 발간개요V. 취약계층VI. 특별사안 II. 요약III. 시민적·정치적 권리\n풀려난 경우도 있었지만, 중국 체류기간이 긴 경우에는 노동교화형\n까지 선고받은 경우도 있었다.\n북한에서는 공정한 재판을 받을 권리가 제대로 보장되지 않는 것\n으로 나타났다. 행정기관도 징역형의 성격을 갖는 노동교양처벌을 \n부과할 수 있으며, 반국가·반민족 범죄행위를 한 경우에는 법원에 \n의한 재판 없이 정치범수용소에 수용될 수 있다. 북한에서는 법원의 \n독립성도 인정되지 않고 있다. ‘모든 국가기관은 조선노동당의 영도 \n밑에 활동이 진행된다’는 북한 사회주의헌법에 따라 사법부는 실질\n적으로 조선노동당의 통제를 받고 있다. 또한, 북한에서는 공개재판\n제도가 주민교양을 위한 선전도구로 이용되고 있는 것으로 나타났\n다. 특별히 본

#위 pipeline 하나의 함수로 제작하기

In [140]:
def HYDE_RAG(question):

  llm = ChatOpenAI(model = 'gpt-4.1', temperature = 0)

  #1. 가상 문서 작성
  virtual_doc_prompt_template = ChatPromptTemplate.from_template("""
  당신은 고도로 숙련된 문서 작성가입니다.
  아래의 질문에 직접적인 답변을 제시하는 짧은 문서를 작성하세요.
  문서의 길이는 200자 내외로 하세요.

  질문 : {question}
  """)
  prompt = virtual_doc_prompt_template.format_messages(question = question)
  answer = llm.invoke(prompt)

  virtual_doc = answer.content

  print('1. 가상 문서 작성')
  print('virtual_doc : ', virtual_doc)
  print('-' * 50)

  #2. 가상 문서 기반 VectorDB 검색
  docs = vector_db.similarity_search(virtual_doc)
  context = "\n\n".join(doc.page_content for doc in docs)

  print('2. 가상 문서 기반 VectorDB 검색')
  print('context : ', context)
  print('-' * 50)


  #3. 최종 답변 생성
  prompt_template = ChatPromptTemplate.from_template("""
  다음 검색 결과를 참고하여 질문에 대한 답변을 작성하세요.
  검색 결과에 질문에 대한 답변이 없다면 자료에 질문에 대한 답이 없다고 답변하세요.

  검색 결과 : {context}

  질문 : {question}

  답변 :
  """)
  prompt = prompt_template.format_messages(context = context, question = question)
  answer = llm.invoke(prompt)

  print('3. 최종 답변 생성')
  print('answer : ', answer.content)
  print('-' * 50)


  return answer.content



In [141]:
result = HYDE_RAG('북한의 군사력 수준은?')

1. 가상 문서 작성
virtual_doc :  북한은 세계에서 군사력이 강한 국가 중 하나로 평가받습니다. 약 120만 명의 현역 군인과 대규모 예비군을 보유하고 있으며, 핵무기와 탄도미사일 등 비대칭 전력도 갖추고 있습니다. 다만, 장비의 현대화 수준은 낮은 편입니다.
--------------------------------------------------
2. 가상 문서 기반 VectorDB 검색
context :  2023 북한인권보고서
442
다. 그중 91,777명이 사망하여 현재 생존자는 41,900명이며, 생존
자의 85.5%는 70세 이상의 고령이다. 425 이산의 계기 또는 사유는 
다양하다. 해방 이후 38도선의 확정에 따른 남북 왕래 차단, 북한의 
공산화 과정에서 월남, 한국전쟁 기간에 월남·월북 및 납치, 의용군 
입대, 정전협정 체결 이후 미귀환(미송환), 북한 이탈 등이 그것이다. 
(1) 월남자 가족
월남자는 한국전쟁에 인민군으로 참여하였다가 행방불명 또는 
전사자로 처리되었다가 한국 또는 제3국에 거주하는 것으로 확인
된 경우, 한국에서 피신이나 임시거주 등의 이유로 북한으로 돌아가
지 못한 경우이다. 북한 당국은 월남자 가족을 복잡한군중으로 분류
하여 관리하고 있는 것으로 보이지만, 차별의 정도를 달리하고 있는 
것으로 파악되었다. 
실제로 월남자 가족의 대학진학, 입대 등 사회적 지위 획득 과정
에서의 차별이 일괄적으로 이뤄진 것은 아니었던 것으로 보인다. 인
민군으로 전쟁에 참여하였던 월남자의 가족은 입당은 가능하였지
만, ‘당일꾼’, ‘법일꾼(안전원, 보위원)’ 등이 될 수 없었다는 진술이 
있었다. 군제대 후 입당을 한 상태에서 보위원이 되기 위해 ‘보위대
학’ 입학 추천을 받을 예정이었으나, 인민군으로 참전한 큰아버지
가 돌아오지 않은 이유가 해명되지 않아 추천이 무산되었다고 한다. 
하지만 인민위원회에서 운영하는 노동교양대의 대장으로 배치되어 
‘행정일꾼’으로 일하였다는 사례가 있었다. 인민군으로 전쟁에 참여
42